# C.5 花更多計算值得嗎？


一份80-token答案，四份各20-token答案，生成文字都80。若每份還由模型用5個token驗證，前者總85，後者總100。候選多出的是選擇與驗證成本，不是免費品質。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
plans = [
    {"candidates": 1, "answer_tokens_each": 80},
    {"candidates": 4, "answer_tokens_each": 20},
]
verify_tokens_each = 5
for plan in plans:
    n = plan["candidates"]
    generated = n * plan["answer_tokens_each"]
    verified = n * verify_tokens_each
    print("候選", n, "生成", generated, "驗證", verified, "總token", generated + verified)

此例只把人工預算相加，沒有生成、量時間或量正確率。選擇器若是普通算術程式，就沒有新增模型token，但仍佔CPU時間；不能把不同單位的成本直接混加。

回答當下多花的計算稱test-time compute，例如長步驟、多候選與驗證，可保持模型權重固定，只改作答流程。固定總token預算時，候選數多就要少給每份；或增加預算時明說總成本變了，再看三件事：「最終正確」是最後交出的答案答對；「集合命中」是候選中至少一份答對；「選錯」則是集合有正解卻交出錯解。前兩步的區別見[C.3](https://birdhackor.github.io/tiny-perceptron-vlm/C.3.html)與[C.4](https://birdhackor.github.io/tiny-perceptron-vlm/C.4.html)。

token數不是秒數。多份可批次並行，長答案仍逐token接；KV快取省重算，每步仍取用先前內容。等待應從收到問題到交出最後文字量，含輸入整理、生成、將生成ID還原為文字與選擇；只量生成的一段不能冒充完整服務延遲。冷啟動與模型訓練另記，不混在一次使用等待。

舊步驟的候選數k，和程式裡的n一樣，都是每題候選份數。k=8比k=1集合命中多三題，卻多約八倍生成token，而多數決只多一題最終正確。多數決選出現最多次的答案、平票取先出現者，規則見[C.4](https://birdhackor.github.io/tiny-perceptron-vlm/C.4.html)。這個例子說明先說預算，再談收益；它沒有判定等預算下哪種安排最佳。

<details>
<summary>補充：舊候選的生成token、計時範圍與限制</summary>

舊比較每份短答最多新增8個token，每份步驟最多48個，沒有固定相同總生成預算。下表摘出每題1份與8份候選兩種安排，都是全部24題的實際生成總token，包含EOS（表示生成結束的專用token，也計入生成數量），以及各題候選batch生成時間的總和；同題候選一起計算，不能把秒數當作逐份串行成本：

| k | 短答生成token | 步驟生成token | 短答生成秒數 | 步驟生成秒數 |
| --- | --- | --- | --- | --- |
| 1 | 51 | 512 | 0.239 | 1.160 |
| 8 | 400 | 4,132 | 0.120 | 1.190 |

步驟k=8的命中14/24比k=1的11/24多三題，卻用了約八倍生成token；多數決只從11/24升至12/24。這是增加預算的觀察，沒有證明等預算下最佳配方。短答k=8的token較多、時間卻較短，受batch與先後執行條件影響；每格只有一次流程，不能當穩定速度排名或把token比值當時間比值。

計時欄包含模型逐token生成及採樣；沒有包含事先將問題與對話標記整理成輸入ID，也沒有包含事後把生成ID還原成可讀文字、投票或程式驗證。這裡排除的是ID還原文字，不是模型逐token生成的運算。算術驗證沒有新增模型token，仍有CPU工作。

表格只量生成，完整服務等待還要包含前後處理；準備環境、訓練與首次載入模型則是另外的成本。完整比較條件與記錄見[實驗報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/reasoning.json)。

</details>
